# Delta Table Maintenance

This notebook optimizes Delta tables in Silver and Gold layers for improved query performance and storage efficiency.

## Operations
1. **OPTIMIZE**: Compacts small files into larger ones for better query performance
2. **ZORDER**: Co-locates related data for faster filtering on common columns
3. **VACUUM**: Removes old file versions to reclaim storage (7-day retention)

## Schedule
- **OPTIMIZE + ZORDER**: Weekly or when query performance degrades
- **VACUUM**: Monthly or when storage costs increase significantly

## Prerequisites
- Silver and Gold layers must be populated
- Sufficient Fabric capacity for compaction operations

⚠️ **Warning**: VACUUM permanently deletes old file versions. Ensure time travel queries beyond 7 days are not required.

In [ ]:
from pyspark.sql.utils import AnalysisException
import os
from datetime import datetime

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

# Configuration
LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = os.environ.get("SILVER_DB", "ag")
GOLD_DB = os.environ.get("GOLD_DB", "au")
VACUUM_RETENTION_HOURS = int(os.environ.get("VACUUM_RETENTION_HOURS", "168"))  # 7 days default
DRY_RUN = os.environ.get("DRY_RUN", "false").lower() == "true"



print(f"Configuration:")
print(f"  Silver DB: {SILVER_DB}")
print(f"  Gold DB: {GOLD_DB}")
print(f"  Vacuum Retention: {VACUUM_RETENTION_HOURS} hours ({VACUUM_RETENTION_HOURS / 24:.1f} days)")
print(f"  Dry Run: {DRY_RUN}")
print(f"  Start Time: {datetime.now().isoformat()}")
print()

In [ ]:
# SHARED SOURCE (generated - do not edit): retail_setup/generation/job_locks.py
# Edit utility/src/retail_setup/generation/job_locks.py, then run utility/scripts/build_notebooks.py.

"""Lease locks that keep Lakehouse writers from overlapping.

``streaming-data-load`` (03, 04), ``daily-maintenance`` (05 and the ML
refresh), ``ml-required`` (setup-11 and the ML producers) and notebook 90 run
on independent schedules and write the same Delta tables. Fabric has no
cross-pipeline mutex, so each writer claims a lease row in
``<silver>._job_locks`` before it writes:

- ``silver``: notebook 03 and notebook 90 (all Silver tables and ``_watermarks``);
- ``gold``: notebook 04 and setup-11 (the ten Gold aggregates, ``setup_run_log``);
- ``ml:<table>``: the producer of that ML output table.

Notebook 05 claims the same lock for each table it maintains, without
waiting, and skips the table when another job holds it.

A claim is a Delta ``MERGE`` that succeeds only when the row is missing,
expired, or already ours; concurrent claims conflict in Delta and one of them
loses. The holder renews its lease from a background heartbeat, so a crashed
notebook releases its locks once the lease expires (``JOB_LOCK_TTL_SECONDS``).
All times come from Spark's ``current_timestamp()``, never the driver clock.

Keep this module free of package imports: Lakehouse notebooks run it as a
cell (``scripts/build_notebooks.py`` copies it in).
"""

import threading
import time
import uuid
from collections.abc import Callable, Iterable
from datetime import datetime, timedelta, timezone

JOB_LOCK_TABLE = "_job_locks"
JOB_LOCK_TTL_SECONDS = 900
JOB_LOCK_HEARTBEAT_SECONDS = 120
JOB_LOCK_POLL_SECONDS = 30
SILVER_LOCK = "silver"
GOLD_LOCK = "gold"


def ml_output_lock(table: str) -> str:
    """The lock that guards one ML output table."""
    return f"ml:{table}"


def is_concurrent_write_error(exc: BaseException) -> bool:
    """True for Delta optimistic-concurrency conflicts (another writer won)."""
    text = f"{type(exc).__name__}: {exc}"
    return "Concurrent" in text or "DELTA_CONCURRENT" in text


class DeltaJobLockStore:
    """Lease rows in a Delta table: (lock_name, owner, acquired_at, expires_at)."""

    def __init__(self, spark, table_name: str, ttl_seconds: int = JOB_LOCK_TTL_SECONDS):
        self.spark = spark
        self.table_name = table_name
        self.ttl_seconds = int(ttl_seconds)
        self.spark.sql(
            f"CREATE TABLE IF NOT EXISTS {table_name} "
            "(lock_name STRING, owner STRING, acquired_at TIMESTAMP, expires_at TIMESTAMP) "
            "USING DELTA"
        )

    @staticmethod
    def _quote(value: str) -> str:
        return "'" + value.replace("\\", "\\\\").replace("'", "\\'") + "'"

    def holder(self, name: str) -> str | None:
        """The owner of a live lease on ``name``, or None."""
        rows = self.spark.sql(
            f"SELECT owner FROM {self.table_name} "
            f"WHERE lock_name = {self._quote(name)} AND expires_at > current_timestamp()"
        ).collect()
        return rows[0]["owner"] if rows else None

    def try_claim(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"""
                MERGE INTO {self.table_name} AS t
                USING (
                    SELECT {self._quote(name)} AS lock_name, {self._quote(owner)} AS owner,
                           current_timestamp() AS acquired_at,
                           current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS AS expires_at
                ) AS s
                ON t.lock_name = s.lock_name
                WHEN MATCHED AND (t.expires_at <= s.acquired_at OR t.owner = s.owner)
                    THEN UPDATE SET t.owner = s.owner, t.acquired_at = s.acquired_at,
                                    t.expires_at = s.expires_at
                WHEN NOT MATCHED THEN INSERT (lock_name, owner, acquired_at, expires_at)
                    VALUES (s.lock_name, s.owner, s.acquired_at, s.expires_at)
                """
            )
        except Exception as exc:
            if is_concurrent_write_error(exc):
                return False
            raise
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        try:
            self.spark.sql(
                f"UPDATE {self.table_name} "
                f"SET expires_at = current_timestamp() + INTERVAL {self.ttl_seconds} SECONDS "
                f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
            )
        except Exception as exc:
            if not is_concurrent_write_error(exc):
                raise
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        for _attempt in range(5):
            try:
                self.spark.sql(
                    f"DELETE FROM {self.table_name} "
                    f"WHERE lock_name = {self._quote(name)} AND owner = {self._quote(owner)}"
                )
                return
            except Exception as exc:
                if not is_concurrent_write_error(exc):
                    raise
                time.sleep(1)


class MemoryJobLockStore:
    """The same lease semantics in memory, with an injectable clock (tests)."""

    def __init__(
        self,
        ttl_seconds: int = JOB_LOCK_TTL_SECONDS,
        clock: Callable[[], datetime] | None = None,
    ):
        self.ttl = timedelta(seconds=ttl_seconds)
        self.clock = clock or (lambda: datetime.now(timezone.utc).replace(tzinfo=None))
        self.rows: dict[str, tuple[str, datetime]] = {}

    def holder(self, name: str) -> str | None:
        row = self.rows.get(name)
        return row[0] if row and row[1] > self.clock() else None

    def try_claim(self, name: str, owner: str) -> bool:
        now = self.clock()
        row = self.rows.get(name)
        if row is None or row[1] <= now or row[0] == owner:
            self.rows[name] = (owner, now + self.ttl)
        return self.holder(name) == owner

    def renew(self, name: str, owner: str) -> bool:
        row = self.rows.get(name)
        if row is not None and row[0] == owner:
            self.rows[name] = (owner, self.clock() + self.ttl)
        return self.holder(name) == owner

    def release(self, name: str, owner: str) -> None:
        if self.rows.get(name, ("", None))[0] == owner:
            del self.rows[name]


def new_lock_owner(holder: str) -> str:
    """A unique owner for one notebook run, e.g. ``03-streaming-to-silver:1a2b...``."""
    return f"{holder}:{uuid.uuid4().hex[:12]}"


def acquire_job_locks(
    store,
    names: Iterable[str],
    owner: str,
    *,
    wait_seconds: float,
    poll_seconds: float = JOB_LOCK_POLL_SECONDS,
    sleep: Callable[[float], None] = time.sleep,
    monotonic: Callable[[], float] = time.monotonic,
) -> list[str] | None:
    """Claim every lock in ``names`` or none of them.

    Locks are claimed in sorted order, so two jobs that need several locks
    can't deadlock. Retries until ``wait_seconds`` elapse (0 = one attempt)
    and returns the claimed names, or None when another job still holds one.
    """
    ordered = sorted(set(names))
    deadline = monotonic() + max(wait_seconds, 0)
    while True:
        claimed = []
        for name in ordered:
            if not store.try_claim(name, owner):
                break
            claimed.append(name)
        if len(claimed) == len(ordered):
            return claimed
        for name in reversed(claimed):
            store.release(name, owner)
        remaining = deadline - monotonic()
        if remaining <= 0:
            return None
        sleep(min(poll_seconds, remaining))


class JobLocks:
    """A notebook's set of leases, renewed by a heartbeat thread until released."""

    def __init__(
        self,
        store,
        holder: str,
        names: Iterable[str],
        heartbeat_seconds: float = JOB_LOCK_HEARTBEAT_SECONDS,
    ):
        self.store = store
        self.owner = new_lock_owner(holder)
        self.names = sorted(set(names))
        self.heartbeat_seconds = heartbeat_seconds
        self.held: list[str] = []
        self.lost: list[str] = []
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None

    def blockers(self) -> dict[str, str | None]:
        """Current holders of the locks this job needs (for messages)."""
        return {name: self.store.holder(name) for name in self.names}

    def acquire(self, wait_seconds: float, **kwargs) -> bool:
        claimed = acquire_job_locks(
            self.store, self.names, self.owner, wait_seconds=wait_seconds, **kwargs
        )
        if claimed is None:
            return False
        self.held = claimed
        if self.heartbeat_seconds > 0:
            self._thread = threading.Thread(target=self._heartbeat, daemon=True)
            self._thread.start()
        return True

    def _heartbeat(self) -> None:
        while not self._stop.wait(self.heartbeat_seconds):
            self.renew()

    def renew(self) -> None:
        for name in self.held:
            try:
                renewed = self.store.renew(name, self.owner)
            except Exception as exc:
                print(f"Lock {name}: renewal failed ({exc}); retrying at the next heartbeat")
                continue
            if not renewed and name not in self.lost:
                self.lost.append(name)

    def assert_held(self) -> None:
        """Fail before a commit point if a lease expired and another job took it."""
        if self.lost:
            raise RuntimeError(f"Job locks lost before completion: {self.lost}")

    def release(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=self.heartbeat_seconds + 5)
        for name in reversed(self.held):
            self.store.release(name, self.owner)
        self.held = []

In [ ]:
# Writer locks (job_locks): each table is maintained only while this notebook
# holds the lock its writers use; a table locked by another job is skipped.
GOLD_AGGREGATE_TABLES = {
    "sales_minute_store", "top_products_15m", "inventory_position_current",
    "dc_inventory_position_current", "truck_dwell_daily", "online_sales_daily",
    "zone_dwell_minute", "marketing_cost_daily", "campaign_performance_daily",
    "tender_mix_daily",
}
LOCK_STORE = DeltaJobLockStore(spark, f"{LAKEHOUSE_NAME}.{SILVER_DB}.{JOB_LOCK_TABLE}")
MAINTENANCE_OWNER = new_lock_owner("05-maintain-delta-tables")


def table_lock_name(db, table):
    """The writer lock for a table, or None when no notebook job writes it."""
    if table == JOB_LOCK_TABLE or table.startswith("_stream_state_"):
        # The lock table itself, and the live stream's own state (the stream
        # isn't a pipeline job; a conflicting commit is skipped below).
        return None
    if table == "setup_run_log" or table in GOLD_AGGREGATE_TABLES:
        return GOLD_LOCK
    if db == SILVER_DB:
        return SILVER_LOCK
    return ml_output_lock(table)


def run_locked(db, table, operation):
    """Run one maintenance operation unless another job holds the table's lock."""
    lock_name = None if DRY_RUN else table_lock_name(db, table)
    if lock_name is not None and acquire_job_locks(
        LOCK_STORE, [lock_name], MAINTENANCE_OWNER, wait_seconds=0
    ) is None:
        holder = LOCK_STORE.holder(lock_name)
        print(f"    - Locked by {holder} ({lock_name}); skipping: {db}.{table}")
        return None
    try:
        return operation()
    finally:
        if lock_name is not None:
            LOCK_STORE.release(lock_name, MAINTENANCE_OWNER)

## Helper Functions

In [ ]:
def optimize_table(db, table, zorder_cols=None):
    return run_locked(db, table, lambda: _optimize_table(db, table, zorder_cols))


def vacuum_table(db, table, retention_hours):
    return run_locked(db, table, lambda: _vacuum_table(db, table, retention_hours))


def _optimize_table(db, table, zorder_cols=None):
    """
    Optimize a Delta table with optional ZORDER.
    
    Args:
        db: Database name
        table: Table name
        zorder_cols: List of columns to ZORDER by (optional)
    """
    full_name = f"{LAKEHOUSE_NAME}.{db}.{table}"
    
    try:
        if not spark.catalog.tableExists(full_name):
            print(f"    - Table not found; skipping: {full_name}")
            return None
        if DRY_RUN:
            print(f"  [DRY RUN] Would optimize {full_name}")
            if zorder_cols:
                print(f"  [DRY RUN] Would ZORDER by: {', '.join(zorder_cols)}")
            return True
        
        # Basic optimization
        if zorder_cols:
            zorder_clause = ", ".join(zorder_cols)
            sql = f"OPTIMIZE {full_name} ZORDER BY ({zorder_clause})"
            print(f"  Optimizing {full_name} with ZORDER by ({zorder_clause})...")
        else:
            sql = f"OPTIMIZE {full_name}"
            print(f"  Optimizing {full_name}...")
        
        result = spark.sql(sql)
        result.collect()
        print(f"    Optimized successfully: {full_name}")
        return True
        
    except Exception as e:
        if is_concurrent_write_error(e):
            print(f"    - Concurrent writer; skipping: {full_name}")
            return None
        print(f"    ✗ Error optimizing {full_name}: {e}")
        return False

def _vacuum_table(db, table, retention_hours):
    """
    Vacuum a Delta table to remove old file versions.
    
    Args:
        db: Database name
        table: Table name
        retention_hours: Hours to retain old versions
    """
    full_name = f"{LAKEHOUSE_NAME}.{db}.{table}"
    
    try:
        if not spark.catalog.tableExists(full_name):
            print(f"    - Table not found; skipping: {full_name}")
            return None
        if DRY_RUN:
            print(f"  [DRY RUN] Would vacuum {full_name} (retain {retention_hours} hours)")
            return True
        
        print(f"  Vacuuming {full_name} (retain {retention_hours} hours)...")
        spark.sql(f"VACUUM {full_name} RETAIN {retention_hours} HOURS")
        print(f"    ✓ Vacuumed successfully")
        return True
        
    except Exception as e:
        if is_concurrent_write_error(e):
            print(f"    - Concurrent writer; skipping: {full_name}")
            return None
        print(f"    ✗ Error vacuuming {full_name}: {e}")
        return False

def get_table_stats(db, table):
    """Get table statistics."""
    try:
        df = spark.table(f"{LAKEHOUSE_NAME}.{db}.{table}")
        return df.count()
    except:
        return None

## Silver Layer Optimization

Optimize Silver fact and dimension tables with ZORDER on frequently filtered columns.

In [ ]:
print("="*80)
print("OPTIMIZING SILVER LAYER")
print("="*80)
print()

# Every Silver table in retail_setup.generation.schemas.TABLES plus the
# streaming and run-control tables written into the Silver schema.
# tests/test_maintenance_table_inventory.py keeps this list equal to the contract.
silver_tables = [
    # Dimensions - no ZORDER needed (small tables)
    ("dim_date", None),
    ("dim_geographies", None),
    ("dim_stores", None),
    ("dim_distribution_centers", None),
    ("dim_trucks", None),
    ("dim_customers", None),
    ("dim_products", None),

    # Streaming control and live-only tables
    ("_watermarks", ["source_table"]),
    # Writer leases (job_locks); see run_locked.
    ("_job_locks", None),
    # Live stream state (store open/closed, visits in progress, batch clock),
    # versioned by checkpoint incarnation and micro-batch.
    ("_stream_state_stores", ["incarnation", "entity_key"]),
    ("_stream_state_visits", ["incarnation", "entity_key"]),
    ("_stream_state_orders", ["incarnation", "entity_key"]),
    ("_stream_state_inventory", ["incarnation", "entity_key"]),
    ("_stream_state_logistics", ["incarnation", "entity_key"]),
    ("_stream_state_meta", ["incarnation", "entity_key"]),
    # Streaming-only operational status; owned by the live projection contract.
    ("fact_online_order_status", ["event_ts", "order_id_ext"]),
    # Append-only setup and Gold publication log (setup notebooks and 04).
    ("setup_run_log", None),

    # Facts - ZORDER by time + primary dimension
    ("fact_receipts", ["event_ts", "store_id"]),
    ("fact_receipt_lines", ["event_ts", "product_id"]),
    ("fact_store_inventory_txn", ["event_ts", "store_id", "product_id"]),
    ("fact_dc_inventory_txn", ["event_ts", "dc_id", "product_id"]),
    ("fact_truck_moves", ["event_ts", "truck_id"]),
    ("fact_truck_inventory", ["event_ts", "truck_id", "product_id"]),
    ("fact_foot_traffic", ["event_ts", "store_id"]),
    ("fact_ble_pings", ["event_ts", "store_id"]),
    ("fact_customer_zone_changes", ["event_ts", "store_id"]),
    ("fact_marketing", ["event_ts", "campaign_id"]),
    ("fact_marketing_attribution", ["purchase_ts", "campaign_id"]),
    ("fact_online_order_headers", ["event_ts", "customer_id"]),
    ("fact_online_order_lines", ["event_ts", "order_id"]),
    ("fact_payments", ["event_ts", "store_id"]),
    ("fact_store_ops", ["event_ts", "store_id"]),
    ("fact_stockouts", ["event_ts", "StoreID", "ProductID"]),
    ("fact_promotions", ["event_ts", "store_id"]),
    ("fact_promo_lines", ["event_ts", "receipt_id_ext"]),
    ("fact_reorders", ["event_ts", "dc_id", "product_id"]),
    ("fact_logistics_incidents", ["started_at", "dc_id"]),
    ("fact_logistics_incident_stores", ["started_at", "store_id"])
]

required_base_tables = [
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.dim_stores",
    f"{LAKEHOUSE_NAME}.{SILVER_DB}.fact_receipts",
    f"{LAKEHOUSE_NAME}.{GOLD_DB}.sales_minute_store",
]
missing_base_tables = [
    table for table in required_base_tables if not spark.catalog.tableExists(table)
]
if missing_base_tables:
    raise RuntimeError(f"Required historical tables are missing: {missing_base_tables}")

silver_success = 0
silver_failed = 0
silver_missing = 0

for table, zorder_cols in silver_tables:
    result = optimize_table(SILVER_DB, table, zorder_cols)
    if result is True:
        silver_success += 1
    elif result is False:
        silver_failed += 1
    else:
        silver_missing += 1

print()
print(f"Silver optimization complete: {silver_success} succeeded, {silver_failed} failed, {silver_missing} skipped (missing or locked)")
print()

## Gold Layer Optimization

Optimize Gold aggregated tables with ZORDER on time-based columns.

In [ ]:
print("="*80)
print("OPTIMIZING GOLD LAYER")
print("="*80)
print()

# Every Gold aggregate (retail_setup.generation.gold.GOLD_TABLES) and every ML
# output in contracts/retail-demo.json. Optional and experimental outputs that
# are not deployed are skipped as missing.
# tests/test_maintenance_table_inventory.py keeps this list equal to the contracts.
gold_tables = [
    ("sales_minute_store", ["ts", "store_id"]),
    ("top_products_15m", ["computed_at", "product_id"]),
    ("inventory_position_current", ["store_id", "product_id"]),
    ("dc_inventory_position_current", ["dc_id", "product_id"]),
    ("truck_dwell_daily", ["day", "site"]),
    ("online_sales_daily", ["day"]),
    ("zone_dwell_minute", ["ts", "store_id"]),
    ("marketing_cost_daily", ["day", "campaign_id"]),
    ("campaign_performance_daily", ["day", "campaign_id"]),
    ("tender_mix_daily", ["day"]),
    # ML outputs (rewritten by each producer run)
    ("demand_forecast", ["store_id", "product_id"]),
    ("customer_segments", None),
    ("churn_predictions", None),
    ("stockout_risk", ["store_id", "product_id"]),
    ("product_associations", None),
    ("product_recommendations", ["product_id"]),
    ("journey_patterns", None),
    ("zone_transitions", None),
    ("zone_dwell_stats", None),
    ("dwell_predictions", None),
    ("price_elasticity", None),
    ("promotion_lift", None),
    ("pricing_constraints", None),
    ("pricing_recommendations", ["product_id"])
]

gold_success = 0
gold_failed = 0
gold_missing = 0

for table, zorder_cols in gold_tables:
    result = optimize_table(GOLD_DB, table, zorder_cols)
    if result is True:
        gold_success += 1
    elif result is False:
        gold_failed += 1
    else:
        gold_missing += 1

print()
print(f"Gold optimization complete: {gold_success} succeeded, {gold_failed} failed, {gold_missing} skipped (missing or locked)")
print()

## Vacuum Old Versions

Remove old file versions to reclaim storage. Default retention is 7 days.

⚠️ **Warning**: This operation is irreversible. Time travel queries beyond the retention period will fail.

In [ ]:
print("="*80)
print("VACUUMING OLD FILE VERSIONS")
print("="*80)
print()

vacuum_success = 0
vacuum_failed = 0
vacuum_missing = 0

# Vacuum Silver tables
print("Vacuuming Silver layer...")
for table, _ in silver_tables:
    result = vacuum_table(SILVER_DB, table, VACUUM_RETENTION_HOURS)
    if result is True:
        vacuum_success += 1
    elif result is False:
        vacuum_failed += 1
    else:
        vacuum_missing += 1

print()

# Vacuum Gold tables
print("Vacuuming Gold layer...")
for table, _ in gold_tables:
    result = vacuum_table(GOLD_DB, table, VACUUM_RETENTION_HOURS)
    if result is True:
        vacuum_success += 1
    elif result is False:
        vacuum_failed += 1
    else:
        vacuum_missing += 1

print()
print(f"Vacuum complete: {vacuum_success} succeeded, {vacuum_failed} failed, {vacuum_missing} skipped (missing or locked)")
print()

## Summary Report

In [ ]:
print("="*80)
print("MAINTENANCE SUMMARY")
print("="*80)
print(f"End Time: {datetime.now().isoformat()}")
print()
print(f"Silver Layer:")
print(f"  Optimized: {silver_success}/{len(silver_tables)} tables")
print()
print(f"Gold Layer:")
print(f"  Optimized: {gold_success}/{len(gold_tables)} tables")
print()
print(f"Vacuum:")
print(f"  Cleaned: {vacuum_success}/{len(silver_tables) + len(gold_tables)} tables")
print()

if DRY_RUN:
    print("⚠️  DRY RUN MODE - No changes were made")
    print("   Set DRY_RUN=false to execute maintenance operations")
else:
    total_failures = silver_failed + gold_failed + vacuum_failed
    if total_failures:
        raise RuntimeError(
            f"Maintenance failed for {total_failures} table operations; "
            "review the per-table errors above"
        )
    print("✓ Maintenance complete")
    print()
    print("Next Steps:")
    print("  1. Monitor query performance for improvements")
    print("  2. Check storage usage reduction")
    print("  3. Schedule this notebook to run weekly for OPTIMIZE")
    print("  4. Schedule this notebook to run monthly for VACUUM")